#### 1. Connection zu SQLite-Datenbank erstellen 

Wir verwenden die Datenbank dbVerin.db. sie enthält die Daten der bekannten Datenbank unseres Sportvereins.

In [1]:
import sqlite3
from py2neo import Graph, Node, Relationship 


conLite = sqlite3.connect('dbVerein.db')


conNeo = Graph("bolt://localhost:7687", auth=("neo4j", "kennwort1"))


# Alle Knoten löschen
conNeo.run("MATCH ()-[r:%]->() DELETE r")
conNeo.run("MATCH (n:%) DELETE n")







(No data)

#### 2. Daten übertragen

Alle Tabellen werden nacheinander aus der SQLite.db geladen und in Knoten und Beziehungen übersetzt.
<ul>
    <li>Der Tabellenname ist das Label des Knotens.</li>
    <li>Jede Entität wird ein Knoten</li>
    <li>Jede Beziehung wird eine Beziehung (dabei sind N:M-Beziehungen in Graphdatenbanken möglich!</li>
    <li>Jede Spalte einer Tabelle wird eine Property beim entsprechenden Knoten.</li>
</ul>

1. Orte

In [2]:
cursor = conLite.cursor()

statement = "SELECT * FROM ort"
cursor.execute(statement)

orte = cursor.fetchall()

for ort in orte:
    nodeOrt = Node("Ort",ort_id = ort[0],plz = ort[1],ortsname = ort[2])
    conNeo.create(nodeOrt)



2. Mitglieder mit Relationship zu Ort (ort_id)

In [3]:
cursor = conLite.cursor()

statement = "SELECT m_id,vorname,nachname,strasse,ort_id FROM mitglied"
cursor.execute(statement)

mitglieder = cursor.fetchall()

for mitglied in mitglieder:
    # Mitglied als Knoten anlegen
    m = Node("Mitglied",m_id = mitglied[0],vorname = mitglied[1],nachname = mitglied[2],strasse = mitglied[3],ort_id = mitglied[4])
    #Mitglied anlegen    
    conNeo.create(m)

# Beziehung zwischen Ort und Mitglied 
linkPersonOrt = "MATCH (p:Mitglied), (o:Ort) WHERE p.ort_id = o.ort_id CREATE (p)-[:WOHNT_IN]->(o)"
conNeo.run(linkPersonOrt)
    

(No data)

3. Sportarten 

In [4]:
cursor = conLite.cursor()

statement = "SELECT sport_id,sportart, beitrag FROM sport"
cursor.execute(statement)

sportarten = cursor.fetchall()

for sportart in sportarten:
    print(sportart)
    # Sportart als Knoten anlegen
    s = Node("Sportart",sport_id = sportart[0],bezeichnung = sportart[1],beitrag = sportart[2])
    # Sportart anlegen    
    conNeo.create(s)
    

(1, 'Fußball', 150)
(2, 'Handball', 110)
(3, 'Turnen', 140)
(4, 'Schwimmen', 160)
(5, 'Boxen', 210)
(6, 'Ringen', 200)
(7, 'Basketball', 120)
(8, 'Judo', 240)
(9, 'Leichtathletik', 180)
(10, 'Volleyball', 240)
(11, 'Aerobic', 200)
(12, 'Badminton', 180)
(13, 'Tennis', 500)
(30000, 'Curling', 100)
(30001, 'Curling2', 60)


4. Relationship Mitglied zu Sportart

In [5]:
cursor = conLite.cursor()

statement = "SELECT m_id, sport_id FROM link_m_s"
cursor.execute(statement)

links = cursor.fetchall()

for link in links:
    # print(link)
    # mitglied und Sportart finden, dann verlinken
    query = "MATCH (m:Mitglied {m_id:"+ str(link[0]) +"}), (s:Sportart {sport_id:" + str(link[1]) + "})"
    query = query + "MERGE (m)-[:IST_REGISTRIERT_BEI]->(s)"
    m = conNeo.run(query)
